# **DEEP LEARNING MODELLING: LSTM MODEL USING GEOGRAPHICAL COORDINATES**

## **1. LIBRARIES**

### **1.1 DEPENDENCIES**

In [ ]:
! pip install adlfs azure-storage-blob
! pip install tensorflow
! pip install tqdm

### **1.2 LIBRARIES**

In [ ]:
# STANDARD LIBRARY
import os
import io
import random

# DATA MANIPULATION AND ANALYSIS
import pandas as pd
import numpy as np

# DATA VISUALISATION
import matplotlib.pyplot as plt
import seaborn as sns

# AZURE CLOUD STORAGE
from adlfs import AzureBlobFileSystem
from azure.storage.blob import BlobServiceClient

# REGULAR EXPRESSIONS
import re

# TIME SERIES
from sklearn.model_selection import TimeSeriesSplit

# STANDARIZATION
from sklearn.preprocessing import StandardScaler

# EVALUATION METRICS
from sklearn.metrics import r2_score

# PROGRESS BAR
from tqdm import tqdm

# TENSORFLOW
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy("mixed_float16")

# SEEDS
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
print(tf.__version__)


2.19.0


In [ ]:
# Correctly set pandas display option for max columns and rows
pd.options.display.max_columns = None
pd.options.display.max_rows = None

## **2. LOADING DATASETS FROM AZURE BLOB STORAGE**

In [ ]:
def load_csv_from_blob(blob_name):
    blob_client = container_client.get_blob_client(blob_name)
    stream = blob_client.download_blob()
    data_bytes = stream.readall()
    data_str = data_bytes.decode("utf-8")
    return pd.read_csv(io.StringIO(data_str))

# Connection configuration
AZURE_STORAGE_ACCOUNT = "researchprojectx24104515"
AZURE_STORAGE_KEY = "bxpexO6i+Hz6n1WiipTn+sTCuLPGMS1BogMERrIrHd16DpQ0GLfQ0R33yrSw4MxsDomq5yNMgw1o+AStlx/MjA=="
connection_string = f"DefaultEndpointsProtocol=https;AccountName={AZURE_STORAGE_ACCOUNT};AccountKey={AZURE_STORAGE_KEY};EndpointSuffix=core.windows.net"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

fs = AzureBlobFileSystem(
    account_name=AZURE_STORAGE_ACCOUNT,
    account_key=AZURE_STORAGE_KEY
)

In [ ]:
# Establish connection to preprocessed dataset
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "preprocessed.csv"
data = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {data.shape}")

Loaded 'preprocessed.csv' with shape: (53424, 44)


In [ ]:
print("Preprocessed Data:")
print(data.shape)
data.head()

Preprocessed Data:
(53424, 44)


,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_rain_int,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
0,2019-01-01 00:00:00,0.0,0.0,12.0,16.0,0.0,0.0,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0.0,9.6,7.8,5.6,9.1,76.0,1035.1,2019,0,0.000000,0,4.0,0,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,0,0,0
1,2019-01-01 01:00:00,0.0,0.0,17.0,14.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,0.0,8.6,7.1,5.3,8.9,79.0,1035.1,2019,0,0.000000,0,3.3,0,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,0,0,0,0
2,2019-01-01 02:00:00,0.0,0.0,18.0,9.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,0.0,8.3,6.9,5.3,8.9,81.0,1034.9,2019,0,8.833333,0,3.0,0,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,0,0,0,0
3,2019-01-01 03:00:00,0.0,0.0,18.0,15.0,0.0,0.0,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,0.0,9.1,7.6,5.8,9.2,79.0,1035.5,2019,0,8.666667,0,3.3,0,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,0,0,0,0
4,2019-01-01 04:00:00,0.0,0.0,12.0,8.0,0.0,0.0,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,0.0,9.2,7.7,5.9,9.3,79.0,1035.7,2019,0,8.866667,0,3.3,0,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,0,0,0,0


In [ ]:
# Establish connection to counters dataset
CONTAINER_NAME = "preprocessed"
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

file_name = "counters.csv"
counters = load_csv_from_blob(file_name)
print(f"Loaded '{file_name}' with shape: {counters.shape}")

Loaded 'counters.csv' with shape: (31, 3)


In [ ]:
print("Counter Locations Data:")
print(counters.shape)
counters

Counter Locations Data:
(31, 3)


,Target,Latitude,Longitude
0,CYC_Clontarf James Larkin Rd,53.368540,-6.170320
1,CYC_Clontarf Pebble Beach Carpark,53.358250,-6.191070
2,CYC_Grove Road Totem,53.329644,-6.270178
3,CYC_North Strand Rd N B,53.358520,-6.241270
4,CYC_Richmond Street Cyclists 1 Merged,53.332160,-6.264400
5,CYC_Richmond Street Cyclists 2 Merged,53.332160,-6.264400
6,FTF_Aston Quay Merged,53.346680,-6.259880
7,FTF_Bachelors Walk Merged,53.347440,-6.260100
8,FTF_Baggot St Lower Merged,53.334540,-6.245620
9,FTF_Baggot St Upper Mespil Rd Bank,53.333770,-6.244510


### **2.1 DEFINE FEATURE CATEGORIES**

In [ ]:
data.head()

,date and time,CYC_Clontarf James Larkin Rd,CYC_Clontarf Pebble Beach Carpark,CYC_Grove Road Totem,CYC_North Strand Rd N B,CYC_Richmond Street Cyclists 1 Merged,CYC_Richmond Street Cyclists 2 Merged,FTF_Aston Quay Merged,FTF_Capel St Mary St Merged,FTF_College St Dame St Merged,FTF_Dame St Merged,FTF_Dolier St Burgh Quay Merged,FTF_Grafton St Monsoon Merged,FTF_Grafton St Nassau St Suffolk St,FTF_Henry St Merged,FTF_Mary St Merged,FTF_Oconnell St Princes St North Merged,FTF_Talbot St Guineys Merged,FTF_Westmoreland St East Fleet St Merged,FTF_Westmoreland St West Merged,WTH_rain,WTH_temp,WTH_wetb,WTH_dewpt,WTH_vappr,WTH_rhum,WTH_msl,TEMP_year,TEMP_holiday,WTH_temp_3h,WTH_rain_int,WTH_dew_spread,WTH_prob_cond,WTH_severe,TEMP_hour_sin,TEMP_hour_cos,TEMP_day_sin,TEMP_day_cos,TEMP_month_sin,TEMP_month_cos,TEMP_weekend,TEMP_mor_peak,TEMP_eve_peak,TEMP_bus_hours
0,2019-01-01 00:00:00,0.0,0.0,12.0,16.0,0.0,0.0,0.0,238.0,0.0,0.0,0.0,140.0,0.0,597.0,163.0,1914.0,0.0,1670.0,1988.0,0.0,9.6,7.8,5.6,9.1,76.0,1035.1,2019,0,0.000000,0,4.0,0,0,0.000000,1.000000,0.781831,0.62349,0.5,0.866025,0,0,0,0
1,2019-01-01 01:00:00,0.0,0.0,17.0,14.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,215.0,0.0,359.0,102.0,885.0,0.0,767.0,1270.0,0.0,8.6,7.1,5.3,8.9,79.0,1035.1,2019,0,0.000000,0,3.3,0,0,0.258819,0.965926,0.781831,0.62349,0.5,0.866025,0,0,0,0
2,2019-01-01 02:00:00,0.0,0.0,18.0,9.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,210.0,0.0,317.0,63.0,984.0,0.0,642.0,1589.0,0.0,8.3,6.9,5.3,8.9,81.0,1034.9,2019,0,8.833333,0,3.0,0,0,0.500000,0.866025,0.781831,0.62349,0.5,0.866025,0,0,0,0
3,2019-01-01 03:00:00,0.0,0.0,18.0,15.0,0.0,0.0,0.0,174.0,0.0,0.0,0.0,204.0,0.0,313.0,59.0,935.0,0.0,582.0,1534.0,0.0,9.1,7.6,5.8,9.2,79.0,1035.5,2019,0,8.666667,0,3.3,0,0,0.707107,0.707107,0.781831,0.62349,0.5,0.866025,0,0,0,0
4,2019-01-01 04:00:00,0.0,0.0,12.0,8.0,0.0,0.0,0.0,82.0,0.0,0.0,0.0,88.0,0.0,172.0,46.0,390.0,0.0,143.0,610.0,0.0,9.2,7.7,5.9,9.3,79.0,1035.7,2019,0,8.866667,0,3.3,0,0,0.866025,0.500000,0.781831,0.62349,0.5,0.866025,0,0,0,0


In [ ]:
# Interaction features created with weather features
interactionf = ["WTH_dew_spread", "WTH_prob_cond", "WTH_severe"]

# Using regex to get all temporal and weather prefix patterns
temp_pattern = re.compile(r"^TEMP_")
wth_pattern = re.compile(r"^WTH_")

# Using regex to get all temporal and weather features
temporalf = [col for col in data.columns if temp_pattern.match(col)]
weatherf = [col for col in data.columns if wth_pattern.match(col) and col not in interactionf]

In [ ]:
weatherf

['WTH_rain',
 'WTH_temp',
 'WTH_wetb',
 'WTH_dewpt',
 'WTH_vappr',
 'WTH_rhum',
 'WTH_msl',
 'WTH_temp_3h',
 'WTH_rain_int']

In [ ]:
temporalf

['TEMP_year',
 'TEMP_holiday',
 'TEMP_hour_sin',
 'TEMP_hour_cos',
 'TEMP_day_sin',
 'TEMP_day_cos',
 'TEMP_month_sin',
 'TEMP_month_cos',
 'TEMP_weekend',
 'TEMP_mor_peak',
 'TEMP_eve_peak',
 'TEMP_bus_hours']

In [ ]:
# Combine all features
fcolumns = weatherf + temporalf + interactionf

# Extract cycle and footfall targets
cycle_t = [col for col in data.columns if col.startswith("CYC_")]
footfall_t = [col for col in data.columns if col.startswith("FTF_")]
targets = cycle_t + footfall_t

# Prepare feature matrix
X = data[fcolumns].copy()
y = data[targets].copy()

# Handle missing values
X = X.ffill().bfill()
y = y.fillna(0)

print(f"Features shape: {X.shape}")
print(f"Targets shape: {y.shape}")
print(f"Feature columns: {len(fcolumns)}")

Features shape: (53424, 24)
Targets shape: (53424, 19)
Feature columns: 24


In [ ]:
targets

['CYC_Clontarf James Larkin Rd',
 'CYC_Clontarf Pebble Beach Carpark',
 'CYC_Grove Road Totem',
 'CYC_North Strand Rd N B',
 'CYC_Richmond Street Cyclists 1 Merged',
 'CYC_Richmond Street Cyclists 2 Merged',
 'FTF_Aston Quay Merged',
 'FTF_Capel St Mary St Merged',
 'FTF_College St Dame St Merged',
 'FTF_Dame St Merged',
 'FTF_Dolier St Burgh Quay Merged',
 'FTF_Grafton St Monsoon Merged',
 'FTF_Grafton St Nassau St Suffolk St',
 'FTF_Henry St Merged',
 'FTF_Mary St Merged',
 'FTF_Oconnell St Princes St North Merged',
 'FTF_Talbot St Guineys Merged',
 'FTF_Westmoreland St East Fleet St Merged',
 'FTF_Westmoreland St West Merged']

# **3. DEEP LEARNING MODELLING : LSTM**

### **3.1 DEFINE LSTM MODEL**

In [ ]:
# Crating LSTM sequences for multiple targets

def lstm_sequences(data, targets, features, counters, seq_len=24):
    # Scale features for all targets
    f_scaler = MinMaxScaler()
    f_data = f_scaler.fit_transform(data[features])

    # Dictionaries to store sequences and scalers
    X_data = {}
    y_data = {}
    t_scalers = {}

    # Ensure Target column in counters has no leading/trailing spaces
    counters["Target"] = counters["Target"].astype(str).str.strip()

    for t in targets:
        # Search for matching target in counters DataFrame
        match = counters.loc[counters["Target"] == t]

        # If no match is found, skip this target
        if match.empty:
            print(f"Warning: Target '{t}' not found in counters DataFrame. Skipping...")
            continue

        # Get latitude and longitude for the target
        lat = match["Latitude"].values[0]
        lon = match["Longitude"].values[0]

        # Scale target values
        y = data[t].values.reshape(-1, 1)
        t_scaler = MinMaxScaler()
        y_scaled = t_scaler.fit_transform(y)

        # Create LSTM sequences
        X, y_seq = [], []
        for i in range(seq_len, len(f_data)):
            # Original feature sequence
            seq = f_data[i-seq_len:i]
            # Add same coordinates across the sequence
            coords = np.tile([lat, lon], (seq_len, 1))
            # Combine features and coordinates
            seq_with_coords = np.concatenate([seq, coords], axis=1)
            X.append(seq_with_coords)
            y_seq.append(y_scaled[i])

        # Save results
        X_data[t] = np.array(X, dtype=np.float32)
        y_data[t] = np.array(y_seq, dtype=np.float32)
        t_scalers[t] = t_scaler

    return X_data, y_data, f_scaler, t_scalers



# Time series split

def ts_split(X, y, train_ratio=0.7, val_ratio=0.15):
    # Index for training and validation
    n_samples = len(X)
    train_end = int(n_samples * train_ratio)
    val_end = int(n_samples * (train_ratio + val_ratio))

    # Define train, test and validation sets
    X_train = X[:train_end]
    y_train = y[:train_end]

    X_val = X[train_end:val_end]
    y_val = y[train_end:val_end]

    X_test = X[val_end:]
    y_test = y[val_end:]

    return X_train, X_val, X_test, y_train, y_val, y_test

# LSTM model

def lstm_model(input_shape):
    reg_strength = 1e-4  # regularisation L2 level

    model = Sequential()
    model.add(Input(shape=input_shape))

    # First LSTM layer
    model.add(LSTM(64, return_sequences=True,
                   kernel_regularizer=regularizers.l2(reg_strength)))
    model.add(Dropout(0.3))

    # Second LSTM layer
    model.add(LSTM(32, return_sequences=False,
                   kernel_regularizer=regularizers.l2(reg_strength)))
    model.add(Dropout(0.3))

    # Middle dense layer
    model.add(Dense(64, activation="relu",
                    kernel_regularizer=regularizers.l2(reg_strength)))
    model.add(Dropout(0.2))

    # Output with one neuron for regression
    model.add(Dense(1))

    # Model compilation
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss="mse",
                  metrics=["mae"])

    return model

# Callback: Early stopping if improvement is not seen after 5 epochs
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

### **3.2 TRAIN MODEL**

In [ ]:
def train_lstm(target_name, X, y, strategy, callbacks=None):
    print(f"\nTraining model for: {target_name}")

    # Splittng data into training, validation, and test sets
    X_train, X_val, X_test, y_train, y_val, y_test = ts_split(X, y)

    # Distribution startegy employed
    print(f"Using strategy: {strategy.__class__.__name__}")

    # Build the LSTM model
    with strategy.scope():
        model = lstm_model(input_shape=X_train.shape[1:])

        # If there are no callbacks defined, this will be as default for early stopping and learning rate
        if callbacks is None:
            early_stopping = EarlyStopping(
                monitor="val_loss",
                # Stop if improvement is not seen after 10 epochs
                patience=10,
                restore_best_weights=True
                )
            reduce_lr = ReduceLROnPlateau(
                monitor="val_loss",
                # Reduce learning rate to half of the value
                factor=0.5,
                # Stop if improvement is not seen after 5 epochs
                patience=5,
                # Do not reduce learning rate below this value
                min_lr=1e-6
                )
            callbacks = [early_stopping, reduce_lr]

        # Training the model
        history = model.fit(
            X_train, y_train,
            # Max epochs allowed
            epochs=100,
            # Samples number per training batch
            batch_size=1024,
            # Validation data for monitoring
            validation_data=(X_val, y_val),
            # Defined callbacks to use
            callbacks=callbacks,
            verbose=1
        )

    print(f"------Training completed for: {target_name}")
    return model, history, X_test, y_test

### **3.3 MODEL VALUATION METRICS**

In [ ]:
def evaluate_mtarget(model, X_test, y_test, target_scaler, target_name, target_idx, n_targets):
    print(f"\n----------- Evaluating model for: {target_name} ({target_idx + 1}/{n_targets})")

    # Predictions
    y_pred = model.predict(X_test)

    # Verifying if y_test is in normalised scale
    if np.max(y_test) <= 1.0:
        y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))
    else:
        print("Warning: y_test seems to be scaled. Evaluation metrics could be inconsistent.")
        y_true = y_test.reshape(-1, 1)

    # Descale predictions
    y_pred = target_scaler.inverse_transform(y_pred)
    y_pred = np.round(y_pred).astype(int)

    # Evaluation metrics
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    mae = np.mean(np.abs(y_true - y_pred))

    # Avoid division by zero in MAPE
    mask = y_true != 0
    if np.any(mask):
        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    else:
        mape = np.nan  # Unable to calculate MAPE

    r2 = r2_score(y_true, y_pred)

    # Results
    print("Evaluation Metrics:")
    print(f" -RMSE : {rmse:.2f}")
    print(f" -MAE  : {mae:.2f}")
    print(f" -MAPE : {mape:.2f}%")
    print(f" -R²   : {r2:.4f}")

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2
    }

### **3.4 APPLYING LSTM**

In [ ]:
# Hours during the day
sequence_length = 24

X_lstm_dict, y_lstm_dict, feature_scaler_lstm, target_scaler_dict = lstm_sequences(
    pd.concat([X, y], axis=1),
    targets,
    fcolumns,
    counters,
    sequence_length
)

strategy = tf.distribute.get_strategy()
print("Shape of a sequence with coordinates:", X_lstm_dict[targets[0]].shape)

Shape of a sequence with coordinates: (53400, 24, 26)


In [ ]:
models = {}
histories = {}
results = {}
test_sets = {}

n_targets = len(targets)

for idx, target_name in enumerate(targets, 1):
    print(f"\n--- Processing target {idx} out of {n_targets}: {target_name}")

    # Get specific data for target
    X_data = X_lstm_dict[target_name]
    y_data = y_lstm_dict[target_name]

    # Training model for target
    model, history, X_test, y_test = train_lstm(
        target_name,
        X_data,
        y_data,
        strategy,
        callbacks=[early_stop]
    )

    # Save model and historical
    models[target_name] = model
    histories[target_name] = history

    # Save test sets for further evaluation
    test_sets[target_name] = (X_test, y_test)

    # Target scaler for predictions and real values descalation
    target_scaler = target_scaler_dict[target_name]

    # Evaluate the model
    metrics = evaluate_mtarget(
        model,
        X_test,
        y_test,
        target_scaler,
        target_name,
        idx,
        n_targets
    )

    results[target_name] = metrics

print("\nTraining and evaluation completed for all targets.")



--- Processing target 1 out of 19: CYC_Clontarf James Larkin Rd

Training model for: CYC_Clontarf James Larkin Rd
Using strategy: _DefaultDistributionStrategy
Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 4s 48ms/step - loss: 0.0382 - mae: 0.1105 - val_loss: 0.0231 - val_mae: 0.0659
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0200 - mae: 0.0600 - val_loss: 0.0205 - val_mae: 0.0670
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0167 - mae: 0.0576 - val_loss: 0.0151 - val_mae: 0.0608
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0138 - mae: 0.0506 - val_loss: 0.0121 - val_mae: 0.0560
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0117 - mae: 0.0442 - val_loss: 0.0136 - val_mae: 0.0738
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0100 - mae: 0.0383 - val_loss: 0.0108 - val_mae: 0.0590
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0089 - mae: 0.0351 - val_loss: 0.0100 - val_mae: 0.0529
Epoch 8/100
37

### **3.5 MODEL SUMMARY**

In [ ]:
for target_name, model in models.items():
    print(f"\n--- Summary for target: {target_name}")
    model.summary()


--- Summary for target: CYC_Clontarf James Larkin Rd


Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_38 (LSTM)                  │ (None, 24, 64)         │        23,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_57 (Dropout)            │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_39 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_58 (Dropout)            │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_59 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,673 (444.05 KB)

 Trainable params: 37,889 (148.00 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_74 (LSTM)                  │ (None, 24, 64)         │        23,296 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_111 (Dropout)           │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_75 (LSTM)                  │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_112 (Dropout)           │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_74 (Dense)                │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_113 (Dropout)           │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_75 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 113,673 (444.05 KB)

 Trainable params: 37,889 (148.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 75,784 (296.04 KB)

### **3.6 RESULTS**

In [ ]:
results = pd.DataFrame.from_dict(results, orient="index")
results.index.name = "Target"
results = results.reset_index()

In [ ]:
print("\nSummary of results LSTM with coordinates:")
results


Summary of results LSTM with coordinates:


,Target,RMSE,MAE,MAPE,R2
0,CYC_Clontarf James Larkin Rd,28.038978,21.949563,380.815510,0.468470
1,CYC_Clontarf Pebble Beach Carpark,74.996888,63.048315,657.111993,-0.871702
2,CYC_Grove Road Totem,74.741013,41.915356,144.178066,0.609028
3,CYC_North Strand Rd N B,5.043085,4.923346,NaN,0.000000
4,CYC_Richmond Street Cyclists 1 Merged,46.533369,35.262047,264.490548,-0.016047
5,CYC_Richmond Street Cyclists 2 Merged,37.385669,31.329713,248.341429,0.227783
6,FTF_Aston Quay Merged,3297.920323,3029.949439,594.334387,-6.033072
7,FTF_Capel St Mary St Merged,1146.372930,972.622097,429.910520,-2.713268
8,FTF_College St Dame St Merged,216.142719,143.012360,80.163863,0.738126
9,FTF_Dame St Merged,534.659940,412.412609,356.388842,-1.801352


### **3.7 PREDICTIONS**

In [ ]:
predictions = {}

for target_name in targets:
    print(f"\nPredicting values for: {target_name}")

    # Test data and targets
    X_test, y_test = test_sets[target_name]
    target_scaler = target_scaler_dict[target_name]

    # Scaled predictions
    y_pred_scaled = models[target_name].predict(X_test)

    # Descale predictions to original values
    y_pred = target_scaler.inverse_transform(y_pred_scaled)
    y_true = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    # DATE AND TIME FROM TEST SET

    # Length used in ltsm sequences
    seq_len = sequence_length

    # Index from start of test set to match the target
    n_total = len(y_lstm_dict[target_name])
    n_train = int(n_total * 0.7)
    n_val   = int(n_total * 0.15)
    start_test_idx = seq_len + n_train + n_val

    datest = data["date and time"].iloc[start_test_idx:].reset_index(drop=True)

    # New DataFrame with values needed for comparison
    predsDataFrame = pd.DataFrame({
        "date and time": datest,
        "Real": y_true.flatten(),
        "LSTM_coords": y_pred.flatten()
    })

    # Round predictions and avoid negative values
    predsDataFrame["LSTM_coords"] = np.round(predsDataFrame["LSTM_coords"]).astype(int)
    predsDataFrame["LSTM_coords"] = predsDataFrame["LSTM_coords"].clip(lower=0)

    predictions[target_name] = predsDataFrame
    print(predsDataFrame.head())

# DataFrame for all predictions
allpreds = pd.concat(
    [dataframe.assign(Target=target) for target, dataframe in predictions.items()],
    ignore_index=True
)


Predicting values for: CYC_Clontarf James Larkin Rd
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
         date and time  Real  LSTM_coords
0  2024-03-07 06:00:00  20.0           25
1  2024-03-07 07:00:00  49.0           35
2  2024-03-07 08:00:00  70.0           43
3  2024-03-07 09:00:00  51.0           43
4  2024-03-07 10:00:00  31.0           38

Predicting values for: CYC_Clontarf Pebble Beach Carpark
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
         date and time   Real  LSTM_coords
0  2024-03-07 06:00:00   26.0           63
1  2024-03-07 07:00:00   75.0           84
2  2024-03-07 08:00:00  105.0           97
3  2024-03-07 09:00:00   75.0           95
4  2024-03-07 10:00:00   33.0           82

Predicting values for: CYC_Grove Road Totem
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
         date and time   Real  LSTM_coords
0  2024-03-07 06:00:00  125.0           53
1  2024-03-07 07:00:00  309.0          160
2  2024-03-07 08:00:00  607.0          286
3  2024-03-07 09:00:00  247.0          

In [ ]:
allpreds.head(20)

,date and time,Real,LSTM_coords,Target
0,2024-03-07 06:00:00,20.0,25,CYC_Clontarf James Larkin Rd
1,2024-03-07 07:00:00,49.0,35,CYC_Clontarf James Larkin Rd
2,2024-03-07 08:00:00,70.0,43,CYC_Clontarf James Larkin Rd
3,2024-03-07 09:00:00,51.0,43,CYC_Clontarf James Larkin Rd
4,2024-03-07 10:00:00,31.0,38,CYC_Clontarf James Larkin Rd
5,2024-03-07 11:00:00,32.0,36,CYC_Clontarf James Larkin Rd
6,2024-03-07 12:00:00,36.0,37,CYC_Clontarf James Larkin Rd
7,2024-03-07 13:00:00,33.0,39,CYC_Clontarf James Larkin Rd
8,2024-03-07 14:00:00,23.0,42,CYC_Clontarf James Larkin Rd
9,2024-03-07 15:00:00,25.0,45,CYC_Clontarf James Larkin Rd


# **4. RESULTS TO AZURE BLOB STORAGE**

In [ ]:
def save_blob(data, filename):
    try:
        blob_name = f"{CONTAINER_NAME}/{filename}"
        csv_data = data.to_csv(index=False)
        with fs.open(blob_name, "w") as f:
            f.write(csv_data)
        print(f"Saved to {blob_name}")
    except Exception as e:
        print(f"Error saving data to blob storage: {str(e)}")
        return False

In [ ]:
CONTAINER_NAME = "results"

save_blob(results, "rlstm_coords.csv")

Saved to results/rlstm_coords.csv


In [ ]:
CONTAINER_NAME = "predictions"

save_blob(allpreds, "pred_lstm_coords.csv")

Saved to predictions/pred_lstm_coords.csv
